In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 📋 Setup & Configuration
# MAGIC 
# MAGIC This notebook configures the environment and installs required packages

# COMMAND ----------

# MAGIC %pip install plotly==5.15.0 requests==2.31.0 python-dateutil==2.8.2

# COMMAND ----------

# Import libraries
import pandas as pd
import numpy as np
import requests
import json
from datetime import datetime, timedelta
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
import matplotlib.pyplot as plt

# COMMAND ----------

# MAGIC %md
# MAGIC ## Google Maps API Configuration

# COMMAND ----------

# Store your Google Maps API key in Databricks secrets
# Go to Workspace -> Admin -> Secrets to set this up
# For demo purposes, we'll use a direct variable (not recommended for production)

# Set your Google Maps API key here
GOOGLE_MAPS_API_KEY = "Your API key here"

# COMMAND ----------

# Configuration variables
class Config:
    GOOGLE_MAPS_API_KEY = GOOGLE_MAPS_API_KEY
    DEPOT_LOCATION = (40.7128, -74.0060)  # New York City
    MAX_VEHICLES = 5
    MAX_PACKAGES = 50

# COMMAND ----------

# MAGIC %md
# MAGIC ## Sample Data Creation

# COMMAND ----------

def create_sample_locations():
    """Create sample customer locations around NYC"""
    locations = [
        {"name": "Times Square", "lat": 40.7589, "lon": -73.9851},
        {"name": "Central Park", "lat": 40.7829, "lon": -73.9654},
        {"name": "Brooklyn Bridge", "lat": 40.7061, "lon": -73.9969},
        {"name": "Empire State", "lat": 40.7484, "lon": -73.9857},
        {"name": "Statue of Liberty", "lat": 40.6892, "lon": -74.0445},
        {"name": "Yankee Stadium", "lat": 40.8296, "lon": -73.9262},
        {"name": "Coney Island", "lat": 40.5755, "lon": -73.9707},
        {"name": "JFK Airport", "lat": 40.6413, "lon": -73.7781},
        {"name": "Wall Street", "lat": 40.7074, "lon": -74.0113},
        {"name": "Metropolitan Museum", "lat": 40.7794, "lon": -73.9632}
    ]
    return pd.DataFrame(locations)

# Create sample data
sample_locations_df = create_sample_locations()
display(sample_locations_df)

# COMMAND ----------

# MAGIC %md
# MAGIC ## Save Configuration

# COMMAND ----------

# Save configuration for other notebooks
dbutils.fs.mkdirs("/mnt/delivery_optimizer/config")

# Save sample locations
spark.createDataFrame(sample_locations_df).write.mode("overwrite").saveAsTable("sample_locations")

name,lat,lon
Times Square,40.7589,-73.9851
Central Park,40.7829,-73.9654
Brooklyn Bridge,40.7061,-73.9969
Empire State,40.7484,-73.9857
Statue of Liberty,40.6892,-74.0445
Yankee Stadium,40.8296,-73.9262
Coney Island,40.5755,-73.9707
JFK Airport,40.6413,-73.7781
Wall Street,40.7074,-74.0113
Metropolitan Museum,40.7794,-73.9632


In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 🗺️ Google Maps Client
# MAGIC 
# MAGIC This notebook handles all Google Maps API interactions

# COMMAND ----------

import requests
import json
from datetime import datetime
import logging

class GoogleMapsDatabricksClient:
    def __init__(self, api_key):
        self.api_key = api_key
        self.base_url = "https://maps.googleapis.com/maps/api"
        self.logger = logging.getLogger(__name__)
    
    def get_distance_matrix(self, origins, destinations, departure_time=None):
        """Get distance matrix between multiple points"""
        try:
            origins_str = "|".join([f"{lat},{lon}" for lat, lon in origins])
            destinations_str = "|".join([f"{lat},{lon}" for lat, lon in destinations])
            
            params = {
                'origins': origins_str,
                'destinations': destinations_str,
                'key': self.api_key,
                'units': 'metric'
            }
            
            if departure_time:
                params['departure_time'] = 'now'
            
            response = requests.get(f"{self.base_url}/distancematrix/json", params=params)
            response.raise_for_status()
            data = response.json()
            
            return data
            
        except Exception as e:
            self.logger.error(f"Error in distance matrix: {e}")
            return self._create_fallback_matrix(origins, destinations)
    
    def get_route_details(self, origin, destination, waypoints=None):
        """Get detailed route information"""
        try:
            origin_str = f"{origin[0]},{origin[1]}"
            destination_str = f"{destination[0]},{destination[1]}"
            
            params = {
                'origin': origin_str,
                'destination': destination_str,
                'key': self.api_key
            }
            
            if waypoints:
                waypoints_str = "|".join([f"{lat},{lon}" for lat, lon in waypoints])
                params['waypoints'] = waypoints_str
            
            response = requests.get(f"{self.base_url}/directions/json", params=params)
            response.raise_for_status()
            data = response.json()
            
            return data
            
        except Exception as e:
            self.logger.error(f"Error getting route details: {e}")
            return None
    
    def _create_fallback_matrix(self, origins, destinations):
        """Create fallback distance matrix using Haversine formula"""
        import math
        
        def haversine(lat1, lon1, lat2, lon2):
            R = 6371  # Earth radius in km
            dlat = math.radians(lat2 - lat1)
            dlon = math.radians(lon2 - lon1)
            a = math.sin(dlat/2) * math.sin(dlat/2) + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon/2) * math.sin(dlon/2)
            c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
            return R * c
        
        matrix = {'rows': []}
        for origin in origins:
            row = {'elements': []}
            for destination in destinations:
                distance_km = haversine(origin[0], origin[1], destination[0], destination[1])
                duration_min = (distance_km / 50) * 60  # Assume 50 km/h
                
                row['elements'].append({
                    'distance': {'text': f'{distance_km:.1f} km', 'value': distance_km * 1000},
                    'duration': {'text': f'{duration_min:.0f} mins', 'value': duration_min * 60},
                    'status': 'OK'
                })
            matrix['rows'].append(row)
        
        return matrix

# COMMAND ----------

# Test the client
if GOOGLE_MAPS_API_KEY != "YOUR_GOOGLE_MAPS_API_KEY_HERE":
    client = GoogleMapsDatabricksClient(GOOGLE_MAPS_API_KEY)
    
    # Test with sample locations
    origins = [(40.7128, -74.0060)]  # NYC
    destinations = [(40.7589, -73.9851)]  # Times Square
    
    result = client.get_distance_matrix(origins, destinations)
    print("Google Maps API Test Result:")
    print(json.dumps(result, indent=2))
else:
    print("⚠️ Please set your Google Maps API key in the Setup notebook")

Google Maps API Test Result:
{
  "destination_addresses": [
    "1556 Broadway, New York, NY 10120, USA"
  ],
  "origin_addresses": [
    "230 Broadway, New York, NY 10007, USA"
  ],
  "rows": [
    {
      "elements": [
        {
          "distance": {
            "text": "9.4 km",
            "value": 9413
          },
          "duration": {
            "text": "22 mins",
            "value": 1305
          },
          "status": "OK"
        }
      ]
    }
  ],
  "status": "OK"
}


In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 📊 Data Models
# MAGIC 
# MAGIC Define data structures and models for the delivery system

# COMMAND ----------

from datetime import datetime, timedelta
from enum import Enum
from pyspark.sql.types import *
from pyspark.sql.functions import *
import numpy as np

class Priority(Enum):
    EXPRESS = 1
    STANDARD = 2
    ECONOMY = 3

class VehicleType(Enum):
    CAR = "car"
    TRUCK = "truck"
    VAN = "van"
    MOTORCYCLE = "motorcycle"

# Define Spark schemas
package_schema = StructType([
    StructField("package_id", StringType(), True),
    StructField("destination_name", StringType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("priority", StringType(), True),
    StructField("weight_kg", DoubleType(), True),
    StructField("volume", DoubleType(), True),
    StructField("time_window_start", TimestampType(), True),
    StructField("time_window_end", TimestampType(), True)
])

vehicle_schema = StructType([
    StructField("vehicle_id", StringType(), True),
    StructField("vehicle_type", StringType(), True),
    StructField("capacity_kg", DoubleType(), True),
    StructField("current_lat", DoubleType(), True),
    StructField("current_lon", DoubleType(), True),
    StructField("avg_speed_kmh", DoubleType(), True),
    StructField("cost_per_km", DoubleType(), True)
])

route_schema = StructType([
    StructField("route_id", StringType(), True),
    StructField("vehicle_id", StringType(), True),
    StructField("package_ids", ArrayType(StringType()), True),
    StructField("total_distance_km", DoubleType(), True),
    StructField("total_duration_min", DoubleType(), True),
    StructField("total_cost", DoubleType(), True),
    StructField("segments", ArrayType(StructType([
        StructField("from_lat", DoubleType(), True),
        StructField("from_lon", DoubleType(), True),
        StructField("to_lat", DoubleType(), True),
        StructField("to_lon", DoubleType(), True),
        StructField("distance_km", DoubleType(), True),
        StructField("duration_min", DoubleType(), True)
    ])), True)
])

# COMMAND ----------

# MAGIC %md
# MAGIC ## Sample Data Generation

# COMMAND ----------

def generate_sample_data(spark, num_packages=20, num_vehicles=3):
    """Generate sample delivery data"""
    
    # Sample locations from our setup
    locations_df = spark.table("sample_locations")
    locations = locations_df.collect()
    
    # Generate packages
    packages_data = []
    for i in range(num_packages):
        loc = locations[i % len(locations)]
        priority = list(Priority)[i % 3]
        
        package = (
            f"PKG{i+1:03d}",
            str(loc["name"]),
            float(loc["lat"]),
            float(loc["lon"]),
            str(priority.name),
            float(round(np.random.uniform(0.5, 10.0), 2)),  # Remove lit(), use float directly
            float(round(np.random.uniform(0.1, 2.0), 2)),   # Remove lit(), use float directly
            None,  # time window start
            None   # time window end
        )
        packages_data.append(package)
    
    packages_df = spark.createDataFrame(packages_data, package_schema)
    
    # Generate vehicles
    vehicles_data = []
    vehicle_types = [VehicleType.TRUCK, VehicleType.VAN, VehicleType.CAR]
    
    for i in range(num_vehicles):
        vtype = vehicle_types[i % len(vehicle_types)]
        capacities = {
            VehicleType.TRUCK: 1000,
            VehicleType.VAN: 500,
            VehicleType.CAR: 100
        }
        speeds = {
            VehicleType.TRUCK: 50,
            VehicleType.VAN: 60,
            VehicleType.CAR: 70
        }
        costs = {
            VehicleType.TRUCK: 1.2,
            VehicleType.VAN: 0.8,
            VehicleType.CAR: 0.5
        }
        
        vehicle = (
            f"VEH{i+1:03d}",
            vtype.name,
            float(capacities[vtype]),
            float(40.7128),  # depot lat
            float(-74.0060), # depot lon
            float(speeds[vtype]),
            float(costs[vtype])
        )


In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 🚚 Route Optimizer
# MAGIC 
# MAGIC Core optimization logic using Google Maps data

# COMMAND ----------

import numpy as np
from pyspark.sql.functions import *

class DatabricksRouteOptimizer:
    def __init__(self, google_maps_client):
        self.client = google_maps_client
        self.depot_location = (40.7128, -74.0060)
    
    def optimize_routes(self, packages_df, vehicles_df, use_real_time=True):
        """Optimize delivery routes using Google Maps data"""
        
        # Convert to pandas for processing
        packages_pd = packages_df.toPandas()
        vehicles_pd = vehicles_df.toPandas()
        
        # Group packages by priority
        express_pkgs = packages_pd[packages_pd['priority'] == 'EXPRESS']
        standard_pkgs = packages_pd[packages_pd['priority'] == 'STANDARD']
        economy_pkgs = packages_pd[packages_pd['priority'] == 'ECONOMY']
        
        optimized_routes = []
        assigned_package_ids = set()
        
        # Assign packages to vehicles
        for _, vehicle in vehicles_pd.iterrows():
            vehicle_id = vehicle['vehicle_id']
            capacity = vehicle['capacity_kg']
            
            # Assign express packages first
            vehicle_packages = []
            current_load = 0
            
            for _, pkg in express_pkgs.iterrows():
                if pkg['package_id'] not in assigned_package_ids and current_load + pkg['weight_kg'] <= capacity:
                    vehicle_packages.append(pkg.to_dict())
                    assigned_package_ids.add(pkg['package_id'])
                    current_load += pkg['weight_kg']
            
            # Then standard packages
            for _, pkg in standard_pkgs.iterrows():
                if pkg['package_id'] not in assigned_package_ids and current_load + pkg['weight_kg'] <= capacity:
                    vehicle_packages.append(pkg.to_dict())
                    assigned_package_ids.add(pkg['package_id'])
                    current_load += pkg['weight_kg']
            
            # Then economy packages
            for _, pkg in economy_pkgs.iterrows():
                if pkg['package_id'] not in assigned_package_ids and current_load + pkg['weight_kg'] <= capacity:
                    vehicle_packages.append(pkg.to_dict())
                    assigned_package_ids.add(pkg['package_id'])
                    current_load += pkg['weight_kg']
            
            if vehicle_packages:
                # Optimize route for this vehicle
                route = self._optimize_vehicle_route(vehicle.to_dict(), vehicle_packages, use_real_time)
                optimized_routes.append(route)
        
        return self._create_route_dataframe(optimized_routes)
    
    def _optimize_vehicle_route(self, vehicle, packages, use_real_time):
        """Optimize route for a single vehicle"""
        
        # Get package locations
        package_locations = [(pkg['latitude'], pkg['longitude']) for pkg in packages]
        
        if not package_locations:
            return {
                'vehicle_id': vehicle['vehicle_id'],
                'packages': [],
                'route': [],
                'total_distance': 0.0,
                'total_duration': 0.0,
                'total_cost': 0.0
            }
        
        # Get distance matrix from Google Maps
        all_points = [self.depot_location] + package_locations
        departure_time = datetime.now() if use_real_time else None
        
        matrix_data = self.client.get_distance_matrix(all_points, all_points, departure_time)
        
        # Find optimal route using nearest neighbor
        route_indices = self._find_optimal_route(matrix_data, len(package_locations))
        
        # Build route segments
        segments = []
        total_distance = 0.0
        total_duration = 0.0
        
        for i in range(len(route_indices) - 1):
            from_idx = route_indices[i]
            to_idx = route_indices[i + 1]
            
            # Validate indices before accessing matrix
            if from_idx < len(matrix_data['rows']) and to_idx < len(matrix_data['rows'][from_idx]['elements']):
                element = matrix_data['rows'][from_idx]['elements'][to_idx]
                
                if element['status'] == 'OK':
                    distance_km = element['distance']['value'] / 1000
                    duration_min = element['duration']['value'] / 60
                    
                    segments.append({
                        'from_lat': all_points[from_idx][0],
                        'from_lon': all_points[from_idx][1],
                        'to_lat': all_points[to_idx][0],
                        'to_lon': all_points[to_idx][1],
                        'distance_km': distance_km,
                        'duration_min': duration_min
                    })
                    
                    total_distance += distance_km
                    total_duration += duration_min
        
        total_cost = total_distance * vehicle['cost_per_km']
        
        return {
            'vehicle_id': vehicle['vehicle_id'],
            'package_ids': [pkg['package_id'] for pkg in packages],
            'total_distance_km': total_distance,
            'total_duration_min': total_duration,
            'total_cost': total_cost,
            'segments': segments
        }
    
    def _find_optimal_route(self, matrix_data, num_locations):
        """Find optimal route using nearest neighbor algorithm"""
        # Start from depot (index 0)
        unvisited = set(range(1, num_locations + 1))  # Package indices
        route = [0]  # Start at depot
        
        while unvisited:
            current = route[-1]
            nearest = None
            min_distance = float('inf')
            
            # Validate current index before accessing matrix
            if current >= len(matrix_data['rows']):
                break
            
            for next_idx in unvisited:
                # Validate next_idx before accessing
                if next_idx < len(matrix_data['rows'][current]['elements']):
                    element = matrix_data['rows'][current]['elements'][next_idx]
                    if element['status'] == 'OK' and element['distance']['value'] < min_distance:
                        min_distance = element['distance']['value']
                        nearest = next_idx
            
            if nearest is not None:
                route.append(nearest)
                unvisited.remove(nearest)
            else:
                break
        
        # Return to depot
        route.append(0)
        return route
    
    def _create_route_dataframe(self, routes):
        """Convert routes to Spark DataFrame"""
        if not routes:
            return spark.createDataFrame([], route_schema)
        
        route_data = []
        for i, route in enumerate(routes):
            route_data.append((
                f"ROUTE{i+1:03d}",
                route['vehicle_id'],
                route['package_ids'],
                route['total_distance_km'],
                route['total_duration_min'],
                route['total_cost'],
                route['segments']
            ))
        
        return spark.createDataFrame(route_data, route_schema)

# COMMAND ----------

# Test the optimizer
if GOOGLE_MAPS_API_KEY != "YOUR_GOOGLE_MAPS_API_KEY_HERE":
    # Load sample data
    packages_df = spark.table("packages")
    vehicles_df = spark.table("vehicles")
    
    # Initialize optimizer
    client = GoogleMapsDatabricksClient(GOOGLE_MAPS_API_KEY)
    optimizer = DatabricksRouteOptimizer(client)
    
    # Run optimization
    optimized_routes = optimizer.optimize_routes(packages_df, vehicles_df)
    
    print("Optimized Routes:")
    display(optimized_routes)
else:
    print("⚠️ Please set your Google Maps API key")

Optimized Routes:


route_id,vehicle_id,package_ids,total_distance_km,total_duration_min,total_cost,segments
ROUTE001,VEH001,"List(PKG016, PKG019, PKG013, PKG001, PKG004, PKG007, PKG010, PKG017, PKG020, PKG011, PKG014, PKG002, PKG005, PKG008, PKG018, PKG012, PKG015, PKG003, PKG006, PKG009)",0.0,0.0,0.0,List()


In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 📊 Delivery Optimization Dashboard (Simplified)
# MAGIC 
# MAGIC Simplified version to ensure it works

# COMMAND ----------

# MAGIC %run "/01_Setup_Config"
# MAGIC %run "/02_Google_Maps_Client"

# COMMAND ----------

# Simplified data creation that definitely works
def create_simple_data(num_packages=5, num_vehicles=2):
    """Create simple test data that definitely works"""
    
    # Simple package data
    packages_data = [
        ("PKG001", "Times Square", 40.7589, -73.9851, "EXPRESS", 5.0, 1.0, None, None),
        ("PKG002", "Central Park", 40.7829, -73.9654, "STANDARD", 3.5, 0.8, None, None),
        ("PKG003", "Brooklyn Bridge", 40.7061, -73.9969, "ECONOMY", 2.0, 0.5, None, None),
        ("PKG004", "Empire State", 40.7484, -73.9857, "EXPRESS", 4.0, 1.2, None, None),
        ("PKG005", "Statue of Liberty", 40.6892, -74.0445, "STANDARD", 6.0, 1.5, None, None)
    ]
    
    # Trim to requested number
    packages_data = packages_data[:num_packages]
    
    # Simple vehicle data
    vehicles_data = [
        ("VEH001", "TRUCK", 1000.0, 40.7128, -74.0060, 50.0, 1.2),
        ("VEH002", "VAN", 500.0, 40.7128, -74.0060, 60.0, 0.8),
        ("VEH003", "CAR", 100.0, 40.7128, -74.0060, 70.0, 0.5)
    ]
    
    # Trim to requested number
    vehicles_data = vehicles_data[:num_vehicles]
    
    packages_df = spark.createDataFrame(packages_data, package_schema)
    vehicles_df = spark.createDataFrame(vehicles_data, vehicle_schema)
    
    return packages_df, vehicles_df

# COMMAND ----------

# Test with simple data first
if GOOGLE_MAPS_API_KEY != "YOUR_GOOGLE_MAPS_API_KEY_HERE":
    print("🔄 Creating sample data...")
    packages_df, vehicles_df = create_simple_data(3, 2)
    
    print("📦 Packages:")
    display(packages_df)
    
    print("🚚 Vehicles:")
    display(vehicles_df)
    
    print("🔄 Running optimization...")
    client = GoogleMapsDatabricksClient(GOOGLE_MAPS_API_KEY)
    optimizer = DatabricksRouteOptimizer(client)
    optimized_routes = optimizer.optimize_routes(packages_df, vehicles_df, True)
    
    print("✅ Optimization Results:")
    display(optimized_routes)
    
else:
    print("⚠️ Please set your Google Maps API key in Notebook 1")


🔄 Creating sample data...
📦 Packages:


package_id,destination_name,latitude,longitude,priority,weight_kg,volume,time_window_start,time_window_end
PKG001,Times Square,40.7589,-73.9851,EXPRESS,5.0,1.0,null,null
PKG002,Central Park,40.7829,-73.9654,STANDARD,3.5,0.8,null,null
PKG003,Brooklyn Bridge,40.7061,-73.9969,ECONOMY,2.0,0.5,null,null


🚚 Vehicles:


vehicle_id,vehicle_type,capacity_kg,current_lat,current_lon,avg_speed_kmh,cost_per_km
VEH001,TRUCK,1000.0,40.7128,-74.006,50.0,1.2
VEH002,VAN,500.0,40.7128,-74.006,60.0,0.8


🔄 Running optimization...
✅ Optimization Results:


route_id,vehicle_id,package_ids,total_distance_km,total_duration_min,total_cost,segments
ROUTE001,VEH001,"List(PKG001, PKG002, PKG003)",39.214,79.11666666666667,47.056799999999996,"List(List(40.7128, -74.006, 40.7061, -73.9969, 1.161, 2.5), List(40.7061, -73.9969, 40.7589, -73.9851, 17.105, 30.566666666666666), List(40.7589, -73.9851, 40.7829, -73.9654, 4.168, 16.2), List(40.7829, -73.9654, 40.7128, -74.006, 16.78, 29.85))"
